In [ ]:
import pandas as pd
import numpy as np

from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, Dropout

from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from tensorflow.keras.callbacks import EarlyStopping


# ------------------------------------------------
# 1 Load Train and Test Data
# ------------------------------------------------
train_data = pd.read_csv("../data/train_data.csv")
test_data = pd.read_csv("../data/test_data.csv")

# ------------------------------------------------
# 2 Define Target Variables
# ------------------------------------------------
targets = ["cpu_percent", "ram_percent", "net_bytes_per_sec"]


# ------------------------------------------------
# 3 Function to Create Sequences
# ------------------------------------------------
def create_sequences(data, targets, window):

    X = []
    y = []

    values = data.values
    target_index = [data.columns.get_loc(col) for col in targets]

    for i in range(window, len(data)):
        X.append(values[i-window:i])
        y.append(values[i, target_index])

    return np.array(X), np.array(y)


# ------------------------------------------------
# 4 Create Time Windows
# ------------------------------------------------
window_size = 10

X_train, y_train = create_sequences(train_data, targets, window_size)
X_test, y_test = create_sequences(test_data, targets, window_size)


print("Train Shape:", X_train.shape)
print("Test Shape:", X_test.shape)


# ------------------------------------------------
# 5 Build LSTM Model
# ------------------------------------------------
model = Sequential()

model.add(LSTM(
    128,
    return_sequences=True,
    input_shape=(X_train.shape[1], X_train.shape[2])
))

model.add(Dropout(0.2))

model.add(LSTM(64))
model.add(Dropout(0.2))

model.add(Dense(3))


# ------------------------------------------------
# 6 Compile Model
# ------------------------------------------------
model.compile(
    optimizer="adam",
    loss="mse"
)


# ------------------------------------------------
# 7 Train Model
# ------------------------------------------------

early_stop = EarlyStopping(
    monitor="val_loss",
    patience=5,
    restore_best_weights=True
)
history = model.fit(
    X_train,
    y_train,
    epochs=50,
    batch_size=32,
    validation_split=0.1,
    callbacks=[early_stop]
)


# ------------------------------------------------
# 8 Predictions
# ------------------------------------------------
y_pred = model.predict(X_test)


# ------------------------------------------------
# 9 Evaluation Metrics
# ------------------------------------------------
mse = mean_squared_error(y_test, y_pred)

rmse = np.sqrt(mse)

mae = mean_absolute_error(y_test, y_pred)

r2 = r2_score(y_test, y_pred)


print("\nLSTM Results")
print("MAE:", mae)
print("MSE:", mse)
print("RMSE:", rmse)
print("R2 Score:", r2)


Train Shape: (21641, 50, 11)
Test Shape: (5373, 50, 11)


/Users/shivanidwivedi/.pyenv/versions/ml_project_env/lib/python3.10/site-packages/keras/src/layers/rnn/rnn.py:199: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


Epoch 1/50
609/609 ━━━━━━━━━━━━━━━━━━━━ 15s 23ms/step - loss: 0.0375 - val_loss: 0.0370
Epoch 2/50
609/609 ━━━━━━━━━━━━━━━━━━━━ 12s 19ms/step - loss: 0.0342 - val_loss: 0.0351
Epoch 3/50
609/609 ━━━━━━━━━━━━━━━━━━━━ 12s 19ms/step - loss: 0.0331 - val_loss: 0.0348
Epoch 4/50
609/609 ━━━━━━━━━━━━━━━━━━━━ 13s 21ms/step - loss: 0.0326 - val_loss: 0.0337
Epoch 5/50
609/609 ━━━━━━━━━━━━━━━━━━━━ 12s 20ms/step - loss: 0.0322 - val_loss: 0.0336
Epoch 6/50
609/609 ━━━━━━━━━━━━━━━━━━━━ 12s 19ms/step - loss: 0.0320 - val_loss: 0.0335
Epoch 7/50
609/609 ━━━━━━━━━━━━━━━━━━━━ 13s 21ms/step - loss: 0.0318 - val_loss: 0.0333
Epoch 8/50
609/609 ━━━━━━━━━━━━━━━━━━━━ 12s 19ms/step - loss: 0.0316 - val_loss: 0.0349
Epoch 9/50
609/609 ━━━━━━━━━━━━━━━━━━━━ 12s 20ms/step - loss: 0.0315 - val_loss: 0.0330
Epoch 10/50
609/609 ━━━━━━━━━━━━━━━━━━━━ 12s 19ms/step - loss: 0.0313 - val_loss: 0.0338
Epoch 11/50
609/609 ━━━━━━━━━━━━━━━━━━━━ 12s 19ms/step - loss: 0.0312 - val_loss: 0.0324
Epoch 12/50
609/609 ━━━━━━━━━━